In [7]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', None)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all" 

In [8]:
embed_dic = {
    'MLP':'EMBED_MLP',
    'GCN2':'EMBED_GCN',
    'SGC2':'EMBED_SGC',
    'FastGCN':'EMBED_FGCN',
    'sga':'sga'
}
atked_dic = {
    'GCN':'ATKED_GCN',
    'RobustGCN':'ATKED_RGCN',
    'GCN_Jaccard':'ATKED_JGCN',
    'SimPGCN':'ATKED_SIMPGCN'
}
we_lr_dic = {
    '5e-5':'5e-5',
    '5e-05':'5e-5',
    '0.05':'5e-2',
    '0.005':'5e-3',
    '0.0005':'5e-4',
    '0.1':'1e-1',
    '0.01':'1e-2',
    '0.001':'1e-3',
    '0.0001':'1e-4'
}
datasets = ['cora', 'citeseer', 'citeseer_full', 'flickr',
                    'cora_full', 'pubmed', 'coauthor_cs', 'coauthor_phy']
added_cols = ['method','atked_model','lay_act','lay_act_cnt','hids_nums','weight_decay','lr']
cols = ['eva_asr','poi_asr','cost','embed_acc','clean_acc']
# cols = ['eva_asr','poi_asr','clean_acc']
def getSplitsC(df_idx):
    splits = []
    for idx in list(df_idx):
        split = idx.split('_')
        if "Jaccard" not in split:
            splits.append(split)
        else:
            splits.append(split[:1] + ['_'.join(split[1:3])] + split[3:])
    return splits
def getSplits(df_idx,cnt=2):
    splits = []
    for idx in list(df_idx):
        split = idx.split('_')
        if 'sga' in split:
            blank = [''] * cnt 
            if "Jaccard" in split:
                splits.append(['sga'] + ['_'.join(split[1:3])] + blank)
            else:
                splits.append(split + blank)
        elif "Jaccard" not in split:
            splits.append([split[0], split[-1]] + split[1:-1])
        else:
            splits.append([split[0]] + ['_'.join(split[-2:])] + split[1:-2])
    return splits
def print_d(df,cnt=2):
    atked_groups = df.groupby('method')
    for embed_model, group_eles in atked_groups:
        g2 = group_eles.groupby('atked_model')
        sga_ = {}
        for atked_model, g2_eles in g2:
            sga_[atked_model] = 0
        for atked_model, g2_eles in g2:
            best = g2_eles.sort_values(by='poi_asr',ascending=False)[:1].values[0]
            if embed_model == 'sga':
                sga_[atked_model] = best[1]
                continue
    print(sga_)
    for embed_model, group_eles in atked_groups:
        g2 = group_eles.groupby('atked_model')
        print('################### start')
        for atked_model, g2_eles in g2:
            best = g2_eles.sort_values(by='poi_asr',ascending=False)[:1].values[0]
            if embed_model == 'sga':
#                 print('poi_asr:',best[1])
                continue
            s00 = best[7]
            s0 = best[8]
#             s1 = 'HIDS[' + str(int(best[9]) - cnt) + ']'
#             s2 = we_lr_dic[best[10]]
#             s3 = we_lr_dic[best[11]]
            print(embed_dic[embed_model], end=' ')
            print(atked_dic[atked_model], end=' ')
            print('['+ ', '.join([s00,s0])+']', ' poi_asr:',best[1], '>=' if float(best[1]) >= float(sga_[atked_model]) else '<', sga_[atked_model])
            print()
        print('################## end')
        print()
def getSplitsD(df_idx):
    splits = []
    for idx in list(df_idx):
        split = idx.split('_')
        if "Jaccard" not in split:
            splits.append(split)
        else:
            splits.append(split[:1] + split[1:-2] + ['_'.join(split[-2:])])
    return splits
def save_test(_prefix, times, seeds):
    print(_prefix)
    tdf = None
    for i in range(times):
        filename = "_".join([_prefix, str(i)]) + '.csv'
        df = pd.read_csv(filename, index_col=0)
        if i == 0:
            tdf = df
        else:
            tdf += df
    tdf /= times
    tdf.index.name = ','.join(np.array(seeds).astype('str'))
    tdf.to_csv(_prefix + '_total.csv')
#     save_test('test_cluster/chameleon_2021_12_02_16_30_01', 5, [2022,2012,1997,5018,2413])
def transform(df_, embed_models, atked_models_):
    print('clean acc:')
    mape = {
        'sga':'SGA',
        'MLP':'C_MLP',
        'SGC2':'C_SGC',
        'GCN2':'C_GCN',
        'FastGCN':'C_FastGCN',
        'MLP&SGC2':'C_MLP&SGC',
        'MLP&GCN2':'C_MLP&GCN'
    }
    embed_models_ = [mape[em] for em in embed_models]
    d = set()
    for i in range(len(df_)):
        atk_model = df_.iloc[i].name[1]
        if atk_model in d:
            continue
        d.add(atk_model)
        print('  {}:{:.3f}'.format(atk_model, df_.iloc[i].clean_acc))
    metrics_ = ['eva_asr', 'poi_asr']
    mapping = {embed_models_[i]:embed_models[i] for i in range(len(embed_models))}
    df = pd.DataFrame(index=embed_models_, columns=pd.MultiIndex.from_product([atked_models_, metrics_]))
    for em in embed_models_:
        ls = []
        for at in atked_models_:
            for asr in metrics_:
                val = df_.loc[mapping[em]].loc[[at]][asr].values[0]
                ls.append(val)
        df.loc[em] = ls
    return df
def get_result(df):
    e_sort_ = ['sga', 'MLP', 'SGC2', 'GCN2', 'FastGCN', 'MLP&SGC2', 'MLP&GCN2']
    er_sort_ = [i for i in range(len(e_sort_))]
    a_sort_ = ['GCN', 'GCN_Jaccard', 'RobustGCN', 'SimPGCN', 'FAGCN', 'H2GCN2', 'H2GCN1', 'SGCPD']
    ar_sort_ = [i for i in range(len(a_sort_))]
    atked_models_ = list(set([idx.split('_')[1] if 'Jaccard' not in idx else '_'.join(idx.split('_')[1:]) for idx in df.index]))
    atked_models_ = sorted(atked_models_, key=lambda x:ar_sort_[a_sort_.index(x)])
    embed_models = list(set([idx.split('_')[0] for idx in df.index]))
    embed_models = sorted(embed_models, key=lambda x:er_sort_[e_sort_.index(x)])
    df = df[cols].sort_values('poi_asr', ascending=False)
    df[['embed_model','atked_model']] = getSplitsD(df.index)
    df = df.set_index(['embed_model','atked_model'])
    df = transform(df, embed_models, atked_models_)
    return df

In [3]:
# save_test('test_cluster/ogbn-arxiv_2021_12_14_20_39_38', 5, [2022,2012,1997,5018,2413])

# 欧氏距离

In [6]:
df = pd.read_csv('test_cluster/cora_2021_12_23_14_23_22_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.846
  GCN_Jaccard:0.834
  RobustGCN:0.829
  SimPGCN:0.828


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.646   0.756       0.616   0.586     0.696   0.762   0.646   
C_MLP        0.43   0.506       0.424   0.364     0.342    0.47   0.412   
C_SGC        0.49   0.602         0.5   0.376      0.47   0.606   0.474   
C_GCN        0.49   0.582       0.506   0.392     0.442   0.542   0.476   
C_FastGCN    0.44   0.492       0.452   0.358     0.334   0.414   0.418   

                   
          poi_asr  
SGA         0.704  
C_MLP       0.462  
C_SGC       0.522  
C_GCN       0.512  
C_FastGCN    0.45

In [62]:
df = pd.read_csv('test_cluster/cora_full_2021_12_24_13_56_01_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.666
  GCN_Jaccard:0.657
  RobustGCN:0.249
  SimPGCN:0.642


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.722   0.732       0.724   0.682      0.35    0.52   0.696   
C_MLP       0.492   0.506       0.516   0.488     0.528   0.538   0.488   
C_SGC       0.398    0.41       0.428    0.41     0.424   0.462   0.262   
C_GCN        0.43   0.452        0.44   0.422     0.408   0.462   0.236   
C_FastGCN    0.38   0.398       0.378   0.376     0.386   0.416   0.208   

                   
          poi_asr  
SGA          0.71  
C_MLP       0.548  
C_SGC       0.294  
C_GCN       0.274  
C_FastGCN    0.26

In [4]:
df = pd.read_csv('test_cluster/citeseer_2021_12_24_00_55_20_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.712
  GCN_Jaccard:0.717
  RobustGCN:0.708
  SimPGCN:0.733


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.52   0.696       0.434   0.558      0.69    0.72   0.364   
C_MLP       0.336   0.428       0.274   0.284     0.386   0.446   0.248   
C_SGC       0.326   0.404       0.292   0.286     0.394   0.456   0.234   
C_GCN       0.316   0.406       0.294     0.3      0.39   0.474   0.242   
C_FastGCN   0.322   0.462       0.284   0.332     0.428    0.51   0.226   

                   
          poi_asr  
SGA         0.486  
C_MLP       0.302  
C_SGC       0.266  
C_GCN       0.272  
C_FastGCN   0.278

In [53]:
df = pd.read_csv('test_cluster/chameleon_2021_12_22_12_27_34_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  GCN_Jaccard:0.429
  RobustGCN:0.529
  SimPGCN:0.399


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.61   0.628       0.592   0.306     0.606   0.632   0.478   
C_MLP       0.578   0.662       0.594   0.338     0.546   0.652    0.57   
C_SGC        0.76   0.764       0.674   0.316      0.68   0.724     0.6   
C_GCN       0.774   0.818       0.594   0.238     0.702    0.77    0.61   
C_FastGCN   0.724    0.78        0.62   0.318     0.696   0.766   0.666   

                   
          poi_asr  
SGA         0.548  
C_MLP       0.642  
C_SGC       0.612  
C_GCN        0.68  
C_FastGCN   0.704

In [199]:
df = pd.read_csv('test_cluster/ogbn-arxiv_2021_12_14_20_39_38_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.675
  GCN_Jaccard:0.675
  RobustGCN:0.449


GCN         GCN_Jaccard         RobustGCN        
      eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr
SGA     0.864   0.862       0.864   0.862     0.554    0.58
C_MLP   0.924   0.916       0.924   0.916     0.654   0.678

In [43]:
df = pd.read_csv('test_cluster/film_2021_12_20_11_13_14_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.273
  GCN_Jaccard:0.290
  RobustGCN:0.268
  SimPGCN:0.287


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.432   0.468       0.368    0.39     0.412   0.414   0.422   
C_MLP       0.398   0.404       0.534   0.426      0.52   0.542   0.526   
C_SGC       0.292   0.298       0.374   0.296     0.358   0.432   0.294   
C_GCN       0.338   0.372       0.416   0.316     0.424   0.482   0.382   
C_FastGCN   0.338   0.354       0.434   0.344     0.478   0.522   0.358   

                   
          poi_asr  
SGA         0.442  
C_MLP       0.514  
C_SGC         0.3  
C_GCN       0.384  
C_FastGCN   0.364

In [19]:
df = pd.read_csv('test_cluster/squirrel_2021_12_20_11_13_14_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.374
  GCN_Jaccard:0.336
  RobustGCN:0.332
  SimPGCN:0.271


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.554   0.558       0.632   0.552     0.546   0.548   0.532   
C_MLP       0.578   0.604       0.612   0.514     0.526   0.592    0.59   
C_SGC       0.564   0.574       0.592   0.488     0.556   0.572   0.588   
C_GCN       0.584   0.588       0.604   0.504     0.562   0.598    0.61   
C_FastGCN   0.518    0.56       0.602   0.448      0.51   0.556    0.42   

                   
          poi_asr  
SGA         0.636  
C_MLP       0.672  
C_SGC       0.638  
C_GCN       0.674  
C_FastGCN   0.542

In [217]:
df = pd.read_csv('test_cluster/blockchain30000_2021_12_27_00_43_00_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD            
            eva_asr     poi_asr
SGA        0.192944  0.00192308
C_MLP      0.580409   0.0144561
C_SGC      0.419734  0.00790367
C_GCN      0.372846   0.0104079
C_FastGCN  0.298681  0.00642136

In [201]:
# mlp+sgc
df = pd.read_csv('test_cluster/cora_2021_12_27_18_28_41_total.csv', index_col=0)
get_result(df)

clean acc:
  RobustGCN:0.829
  GCN:0.846
  SimPGCN:0.828
  GCN_Jaccard:0.834


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.646   0.756       0.616   0.586     0.696   0.762   0.646   
C_MLP&SGC   0.506   0.584       0.498   0.382     0.456    0.57   0.476   

                   
          poi_asr  
SGA         0.704  
C_MLP&SGC    0.52

In [202]:
# mlp+gcn
df = pd.read_csv('test_cluster/cora_2021_12_27_23_51_40_total.csv', index_col=0)
get_result(df)

clean acc:
  RobustGCN:0.829
  GCN:0.846
  SimPGCN:0.828
  GCN_Jaccard:0.834


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.646   0.756       0.616   0.586     0.696   0.762   0.646   
C_MLP&GCN   0.484   0.564       0.486   0.384     0.434   0.534   0.466   

                   
          poi_asr  
SGA         0.704  
C_MLP&GCN   0.496

In [203]:
# mlp+sgc
df = pd.read_csv('test_cluster/chameleon_2021_12_27_18_28_41_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  RobustGCN:0.529
  SimPGCN:0.399
  GCN_Jaccard:0.429


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.61   0.628       0.592   0.306     0.606   0.632   0.478   
C_MLP&SGC    0.74   0.748       0.682   0.342     0.652   0.676   0.662   

                   
          poi_asr  
SGA         0.548  
C_MLP&SGC    0.65

In [204]:
# mlp+gcn
df = pd.read_csv('test_cluster/chameleon_2021_12_27_23_51_40_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  RobustGCN:0.529
  SimPGCN:0.399
  GCN_Jaccard:0.429


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.61   0.628       0.592   0.306     0.606   0.632   0.478   
C_MLP&GCN   0.592   0.674       0.618   0.304      0.57   0.642   0.586   

                   
          poi_asr  
SGA         0.548  
C_MLP&GCN   0.634

In [184]:
df = pd.read_csv('test_cluster/cora_2021_12_30_00_19_17_total.csv', index_col=0)
get_result(df)

clean acc:
  H2GCN2:0.762
  H2GCN1:0.736


H2GCN2          H2GCN1        
          eva_asr poi_asr eva_asr poi_asr
C_GCN       0.198   0.242   0.196   0.228
C_MLP       0.148   0.208   0.156   0.192
SGA         0.262    0.35   0.228   0.272
C_FastGCN    0.16   0.188   0.156   0.172
C_SGC         0.2   0.252     0.2   0.232

In [183]:
df = pd.read_csv('test_cluster/chameleon_2021_12_30_00_19_17_total.csv', index_col=0)
get_result(df)

clean acc:
  H2GCN1:0.403
  H2GCN2:0.421


H2GCN2          H2GCN1        
          eva_asr poi_asr eva_asr poi_asr
C_GCN        0.28   0.314    0.29   0.312
C_MLP       0.236   0.288   0.206   0.292
SGA         0.294   0.316   0.332   0.344
C_FastGCN   0.262   0.294    0.26   0.292
C_SGC       0.278   0.312    0.26   0.276

In [5]:
df = pd.read_csv('test_cluster/chameleon_2021_12_31_12_28_03_total.csv', index_col=0)
get_result(df)

clean acc:
  FAGCN:0.530


FAGCN        
          eva_asr poi_asr
SGA          0.45   0.528
C_MLP        0.34   0.484
C_SGC       0.396    0.49
C_GCN        0.41   0.496
C_FastGCN   0.422   0.524

# wrong_labels

In [9]:
save_test('test_cluster/blockchain50000_2022_01_01_14_56_09', 5, [2022,2012,1997,5018,2413])

test_cluster/blockchain50000_2022_01_01_14_56_09


In [205]:
df = pd.read_csv('test_cluster/cora_2021_12_17_17_28_38_total.csv', index_col=0)
get_result(df)

clean acc:
  RobustGCN:0.829
  GCN:0.846
  SimPGCN:0.828
  GCN_Jaccard:0.834
  FAGCN:0.834


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.646   0.756       0.616   0.586     0.696   0.762   0.646   
C_MLP       0.632   0.722       0.602   0.556     0.666   0.726   0.618   
C_SGC       0.648   0.748       0.604    0.58     0.706   0.766   0.638   
C_GCN        0.65   0.756        0.61   0.582       0.7    0.76   0.636   
C_FastGCN   0.644   0.746       0.612   0.572     0.704   0.764   0.638   

                    FAGCN          
          poi_asr eva_asr poi_asr  
SGA         0.704    0.34    0.48  
C_MLP        0.66    0.33   0.448  
C_SGC       0.682   0.346   0.466  
C_GCN       0.686   0.342   0.464  
C_FastGCN   0.696   0.336   0.452

In [44]:
df = pd.read_csv('test_cluster/cora_full_2021_12_18_22_06_21_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.666
  GCN_Jaccard:0.657
  RobustGCN:0.249
  SimPGCN:0.642


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.722   0.732       0.724   0.682      0.35    0.52   0.696   
C_MLP        0.71   0.718       0.704   0.684     0.344   0.514   0.684   
C_SGC       0.698   0.706       0.692   0.686      0.34   0.508   0.646   
C_GCN       0.686    0.71       0.692    0.68     0.354   0.532   0.636   
C_FastGCN   0.702   0.722       0.706   0.696      0.33   0.536   0.626   

                   
          poi_asr  
SGA          0.71  
C_MLP       0.696  
C_SGC       0.656  
C_GCN       0.648  
C_FastGCN    0.65

In [38]:
df = pd.read_csv('test_cluster/citeseer_2021_12_18_11_00_43_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.712
  GCN_Jaccard:0.717
  RobustGCN:0.708
  SimPGCN:0.733


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.52   0.696       0.434   0.558      0.69    0.72   0.364   
C_MLP       0.506   0.682       0.432   0.538     0.672   0.716   0.306   
C_SGC       0.528   0.706       0.454   0.554      0.67   0.724   0.308   
C_GCN       0.536    0.71        0.45   0.564     0.666   0.712   0.362   
C_FastGCN   0.542   0.698        0.47   0.542     0.638   0.712   0.346   

                   
          poi_asr  
SGA         0.486  
C_MLP       0.458  
C_SGC       0.462  
C_GCN       0.472  
C_FastGCN   0.466

In [55]:
df = pd.read_csv('test_cluster/coauthor_phy_2021_12_21_20_40_55_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.953
  RobustGCN:0.802
  SimPGCN:0.884


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.892   0.892         NaN     NaN     0.342    0.38   0.372   
C_MLP        0.76   0.764         NaN     NaN     0.368   0.396   0.326   
C_SGC       0.776   0.778         NaN     NaN     0.366   0.398   0.362   
C_GCN       0.814   0.816         NaN     NaN     0.344   0.374    0.39   
C_FastGCN   0.834   0.836         NaN     NaN     0.338    0.37   0.368   

                   
          poi_asr  
SGA         0.366  
C_MLP       0.312  
C_SGC       0.354  
C_GCN       0.382  
C_FastGCN   0.354

In [36]:
df = pd.read_csv('test_cluster/chameleon_2021_12_18_11_01_24_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  GCN_Jaccard:0.429
  RobustGCN:0.529
  SimPGCN:0.399


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.61   0.628       0.592   0.306     0.606   0.632   0.478   
C_MLP       0.636    0.63       0.608   0.304     0.552   0.594   0.526   
C_SGC       0.676   0.682        0.63   0.308       0.6   0.668    0.57   
C_GCN        0.64   0.648       0.562   0.288     0.626   0.644   0.508   
C_FastGCN   0.622   0.634       0.566   0.288     0.576   0.616   0.514   

                   
          poi_asr  
SGA         0.548  
C_MLP        0.57  
C_SGC       0.576  
C_GCN       0.566  
C_FastGCN   0.546

In [40]:
df = pd.read_csv('test_cluster/squirrel_2021_12_18_22_04_55_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.374
  GCN_Jaccard:0.336
  RobustGCN:0.332
  SimPGCN:0.271


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.554   0.558       0.632   0.552     0.546   0.548   0.532   
C_MLP       0.552   0.566       0.628   0.526     0.534   0.538   0.594   
C_SGC       0.558   0.558       0.618   0.518      0.49   0.514   0.582   
C_GCN       0.564   0.552       0.604   0.474     0.504   0.528   0.566   
C_FastGCN   0.544   0.558       0.596    0.49     0.526    0.56   0.566   

                   
          poi_asr  
SGA         0.636  
C_MLP       0.666  
C_SGC       0.636  
C_GCN        0.63  
C_FastGCN   0.636

In [41]:
df = pd.read_csv('test_cluster/film_2021_12_19_20_04_12_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.273
  GCN_Jaccard:0.290
  RobustGCN:0.255
  SimPGCN:0.287


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.432   0.468       0.368    0.39     0.258   0.292   0.422   
C_MLP       0.426   0.462       0.376   0.368     0.254   0.282   0.446   
C_SGC       0.394    0.42       0.372   0.338      0.24    0.25     0.4   
C_GCN       0.442   0.466       0.394   0.352     0.244   0.274   0.406   
C_FastGCN    0.41   0.442       0.398    0.35     0.246   0.268    0.41   

                   
          poi_asr  
SGA         0.442  
C_MLP        0.46  
C_SGC       0.418  
C_GCN       0.434  
C_FastGCN   0.428

In [206]:
df = pd.read_csv('test_cluster/ogbn-arxiv_2021_12_23_11_18_31_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.675
  GCN_Jaccard:0.675
  RobustGCN:0.449


GCN         GCN_Jaccard         RobustGCN        
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr
SGA         0.864   0.862       0.864   0.862     0.554    0.58
C_MLP        0.86   0.858        0.86   0.858     0.528   0.548
C_SGC       0.848   0.848       0.848   0.848      0.54   0.536
C_GCN       0.852    0.85       0.852    0.85     0.554   0.538
C_FastGCN   0.844    0.84       0.844    0.84      0.51   0.536

In [218]:
df = pd.read_csv('test_cluster/blockchain30000_2021_12_27_10_47_19_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD            
            eva_asr     poi_asr
SGA        0.192944  0.00192308
C_MLP      0.570427    0.012533
C_SGC      0.444746  0.00790367
C_GCN      0.438984   0.0186724
C_FastGCN  0.387585   0.0123048

In [219]:
df = pd.read_csv('test_cluster/blockchain40000_2021_12_27_13_23_20_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD            
            eva_asr     poi_asr
SGA        0.256688  0.00711154
C_MLP       0.69001   0.0234942
C_SGC      0.618311   0.0233974
C_GCN      0.415313   0.0071446
C_FastGCN  0.299184  0.00360477

In [10]:
df = pd.read_csv('test_cluster/blockchain50000_2022_01_01_14_56_09_total.csv', index_col=0)
get_result(df)

clean acc:
  SGCPD:0.000


SGCPD           
            eva_asr    poi_asr
SGA        0.290756  0.0111266
C_MLP      0.513881  0.0133365
C_SGC      0.565275  0.0255947
C_GCN       0.40525  0.0157838
C_FastGCN  0.465966  0.0201296

In [211]:
# mlp+sgc
df = pd.read_csv('test_cluster/cora_2021_12_27_18_29_41_total.csv', index_col=0)
get_result(df)

clean acc:
  RobustGCN:0.829
  GCN:0.846
  SimPGCN:0.828
  GCN_Jaccard:0.834


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.646   0.756       0.616   0.586     0.696   0.762   0.646   
C_MLP&SGC   0.638   0.734       0.606   0.558      0.67   0.742   0.636   

                   
          poi_asr  
SGA         0.704  
C_MLP&SGC   0.684

In [212]:
# mlp+gcn
df = pd.read_csv('test_cluster/cora_2021_12_27_23_51_14_total.csv', index_col=0)
get_result(df)

clean acc:
  RobustGCN:0.829
  GCN:0.846
  SimPGCN:0.828
  GCN_Jaccard:0.834


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA         0.646   0.756       0.616   0.586     0.696   0.762   0.646   
C_MLP&GCN   0.644   0.744       0.606   0.554     0.676   0.756    0.64   

                   
          poi_asr  
SGA         0.704  
C_MLP&GCN   0.684

In [213]:
# mlp+sgc
df = pd.read_csv('test_cluster/chameleon_2021_12_27_18_29_41_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  RobustGCN:0.529
  SimPGCN:0.399
  GCN_Jaccard:0.429


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.61   0.628       0.592   0.306     0.606   0.632   0.478   
C_MLP&SGC   0.634   0.644        0.64   0.314     0.578   0.618    0.53   

                   
          poi_asr  
SGA         0.548  
C_MLP&SGC    0.54

In [214]:
# mlp+gcn
df = pd.read_csv('test_cluster/chameleon_2021_12_27_23_51_14_total.csv', index_col=0)
get_result(df)

clean acc:
  GCN:0.552
  RobustGCN:0.529
  SimPGCN:0.399
  GCN_Jaccard:0.429


GCN         GCN_Jaccard         RobustGCN         SimPGCN  \
          eva_asr poi_asr     eva_asr poi_asr   eva_asr poi_asr eva_asr   
SGA          0.61   0.628       0.592   0.306     0.606   0.632   0.478   
C_MLP&GCN   0.668   0.664       0.608   0.318     0.618   0.636   0.598   

                   
          poi_asr  
SGA         0.548  
C_MLP&GCN   0.628

In [8]:
# t39
df = pd.read_csv('test_cluster/chameleon_2021_12_30_14_02_13_total.csv', index_col=0)
get_result(df)

clean acc:
  H2GCN2:0.405
  H2GCN1:0.391


H2GCN2          H2GCN1        
          eva_asr poi_asr eva_asr poi_asr
SGA         0.204   0.214   0.214   0.236
C_MLP       0.226    0.24    0.21   0.204
C_SGC       0.222   0.244   0.216   0.214
C_GCN       0.196   0.218   0.202   0.222
C_FastGCN   0.186   0.212     0.2   0.202

In [4]:
# tt
df = pd.read_csv('test_cluster/chameleon_2021_12_30_14_01_43_total.csv', index_col=0)
get_result(df)

clean acc:
  H2GCN1:0.403
  H2GCN2:0.421


H2GCN2          H2GCN1        
          eva_asr poi_asr eva_asr poi_asr
SGA         0.294   0.316   0.332   0.344
C_MLP       0.198   0.204   0.204   0.206
C_SGC         0.2   0.214   0.206   0.214
C_GCN       0.188   0.198    0.19    0.19
C_FastGCN   0.186   0.196   0.206   0.216

In [4]:
# tt
df = pd.read_csv('test_cluster/chameleon_2021_12_29_00_48_40_total.csv', index_col=0)
get_result(df)

clean acc:
  FAGCN:0.530


FAGCN        
          eva_asr poi_asr
SGA         0.454    0.55
C_MLP       0.388    0.48
C_SGC       0.386   0.472
C_GCN       0.422   0.486
C_FastGCN   0.392    0.47

In [6]:
# t39
df = pd.read_csv('test_cluster/chameleon_2021_12_31_13_56_10_total.csv', index_col=0)
get_result(df)

clean acc:
  FAGCN:0.530


FAGCN        
          eva_asr poi_asr
SGA         0.452   0.528
C_MLP       0.414   0.504
C_SGC        0.39    0.51
C_GCN       0.424   0.494
C_FastGCN   0.406   0.468